In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/consolidated_pipeline/1_setup/utils

In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","products","Data Source")


In [0]:
catalog=dbutils.widgets.get('catalog')
data_source=dbutils.widgets.get('data_source')

base_path=f"s3://sportsbar-aws-dp/{data_source}/*.csv"

#Bronze Processing

In [0]:
df=spark.read.format("csv").option("header","true").option("inferSchema","true").load(base_path)\
    .withColumn("read_timestamp",F.current_timestamp())\
    .select("*","_metadata.file_name","_metadata.file_size")

In [0]:
df.write.mode("overwrite").format("delta").option("delta.enableChangeDataFeed","true")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

##Silver Processing

In [0]:
df_bronze=spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source}")
df_bronze.show(10)

##Drop Duplicates

In [0]:
print(f"Before dropping {df_bronze.count()}")
df_silver=df_bronze.dropDuplicates(["product_id"])
print(f"After dropping {df_silver.count()}")

##Title case fix

In [0]:
df_silver.select("category").distinct().show()

In [0]:
df_silver=df_silver.withColumn("category",F.when(F.col('category').isNotNull(),F.initcap("category")).otherwise(None))

Fixing protein spelling

In [0]:
df_silver=df_silver.withColumn("category",F.regexp_replace(F.col("category"),"(?i)protien","Protein"))\
    .withColumn("product_name",F.regexp_replace(F.col("product_name"),"(?i)protien","Protein"))


##Division column

In [0]:
df_silver=df_silver.withColumn("division",F.when(F.col("category")=="Energy Bars","Nutrition Bars")\
                     .when(F.col("category")=="Protein Bars","Nutrition Bars")\
                     .when(F.col("category")=="Granola & Cereals","Breakfast Foods")\
                     .when(F.col("category")=="Recovery Dairy","Dairy & Recovery")\
                     .when(F.col("category")=="Healthy Snacks","Healthy Snacks")\
                     .when(F.col("category")=="Electrolyte Mix","Hydration & Electrolytes")\
                     .otherwise("Other"))

In [0]:
df_silver=df_silver.withColumn("variant",F.regexp_extract(F.col("product_name"),"\((.*?)\)",1))


In [0]:
df_silver.show(10)

In [0]:
df_silver=df_silver.withColumn('product_code',F.sha2(F.col("product_name").cast("string"),256))

In [0]:
df_silver=df_silver.withColumn("product_id",F.when(F.col("product_id")\
    .cast("string").rlike("^[0-9]+$"),F.col("product_id").cast("string")).otherwise(F.lit("99999").cast("string")))


In [0]:
df_silver=df_silver.select("product_code","division","category"\
    ,"product_name","variant","product_id","read_timestamp","file_name","file_size")


In [0]:
df_silver.write.mode("overwrite").format("delta").option("delta.enableChangeDataFeed","true")\
    .option("mergeSchema","true").saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

##Gold processing

In [0]:
df_silver=spark.sql(f"select * from {catalog}.{silver_schema}.{data_source}")
df_gold=df_silver.select("product_code","product_id","division","category","product_name","variant")
df_gold.show(10)

In [0]:
df_gold.write.format("delta").mode("overwrite").option("delta.enableChangeDataFeed","true")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
delta_table=DeltaTable.forName(spark,f"fmcg.gold.dim_products")
df_child_products=spark.sql(f" select product_code,division,category,product_name as product,variant\
     from fmcg.gold.sb_dim_products;")
df_child_products.show(10)

In [0]:
delta_table.alias("t").merge(source=df_child_products.alias("s"),condition="s.product_code=t.product_code")\
    .whenMatchedUpdate(set={"product":"s.product",
                            "category":"s.category",
                            "division":"s.division",
                            "variant":"s.variant"})\
    .whenNotMatchedInsert(values={"product_code":"s.product_code",
                                  "product":"s.product",
                                  "category":"s.category",
                                  "division":"s.division",
                                  "variant":"s.variant"}).execute()
                                  
                

In [0]:
%sql
select * from fmcg.gold.dim_products limit 10